In [1]:
!pip install datasets
!pip install pyarrow

In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
import os
from email import message_from_string
warnings.filterwarnings('ignore')

In [3]:
path = r"D:\Northeastern\Fall2025\DS5500\Spam_Email_Detection\raw_data"

In [4]:
def load_raw_datasets(folder_path):
    data_dict = {}
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(folder_path, file_name)
            try:
                df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
                data_dict[os.path.splitext(file_name)[0]] = df
                print(f"Loaded: {file_name} → {df.shape[0]} rows, {df.shape[1]} columns")
            except Exception as e:
                print(f"Could not load {file_name}: {e}")
    return data_dict

In [5]:
df_nazario = pd.read_csv(os.path.join(path, "Nazario.csv"))
df_nigerian = pd.read_csv(os.path.join(path, "Nigerian_Fraud.csv"))
df_trec_05 = pd.read_csv(os.path.join(path, "TREC_05.csv"), encoding='latin1', engine='python', on_bad_lines='skip')
df_trec_06 = pd.read_csv(os.path.join(path, "TREC_06.csv"), encoding='latin1', engine='python', on_bad_lines='skip')
df_trec_07 = pd.read_csv(os.path.join(path, "TREC_07.csv"), encoding='latin1', engine='python', on_bad_lines='skip')

In [6]:
df_nazario.head(3)

,sender,receiver,date,subject,body,urls,label
0,Mail System Internal Data <MAILER-DAEMON@monke...,NaN,28 Sep 2017 09:57:25 -0400,DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA,This text is part of the internal format of yo...,1,1
1,cPanel <service@cpanel.com>,jose@monkey.org,"Fri, 30 Oct 2015 00:00:48 -0500",Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1,1
2,Microsoft Outlook <recepcao@unimedceara.com.br>,NaN,"Fri, 30 Oct 2015 06:21:59 -0300 (BRT)",Helpdesk Mailbox Alert!!!,Your two incoming mails were placed on pending...,1,1


In [7]:
df_nigerian.head(3)

,sender,receiver,date,subject,body,urls,label
0,MR. JAMES NGOLA. <james_ngola2002@maktoob.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 02:38:20 +0000",URGENT BUSINESS ASSISTANCE AND PARTNERSHIP,FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,0,1
1,Mr. Ben Suleman <bensul2004nng@spinfinder.com>,R@M,"Thu, 31 Oct 2002 05:10:00 -0000",URGENT ASSISTANCE /RELATIONSHIP (P),"Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",0,1
2,PRINCE OBONG ELEME <obong_715@epatra.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 22:17:55 +0100",GOOD DAY TO YOU,FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,0,1


In [8]:
df_trec_05 = df_trec_05[df_trec_05['label'].astype(str).str.strip().isin(['0', '1'])]
df_trec_05['label'] = df_trec_05['label'].astype(int)
df_trec_05['label'] = (
    df_trec_05['label']
    .fillna(0)               
    .astype(str)             
    .str.strip()             
    .replace('', '0')        
    .astype(int)             
)
df_trec_05.head(3)

,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr PW: bnaweb22 -----O...,0,1
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,0
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,1


In [9]:
df_trec_06.head(3)

,sender,receiver,date,subject,body,label,urls
0,jhpb@sarto.budd-lake.nj.us,NaN,"Tue, 28 Jul 1992 03:13:55 +0000",new Catholic mailing list now up and running,The mailing list I queried about a few weeks a...,0.0,0.0
1,Stella Lowry <rookcuduq@yahoo.com>,Brian <bernice@groucho.cs.psu.edu>,"Sat, 03 Apr 1993 10:34:36 -0500",re[12]:,\n ...,1.0,1.0
2,Walter <trwmpca@downtowncumberland.com>,arline@groucho.cs.psu.edu,"Tue, 06 Apr 1993 20:33:13 -0600",Take a moment to explore this.,Academic Qualifications available from prestig...,1.0,0.0


In [10]:
df_trec_07 = df_trec_07[df_trec_07['label'].astype(str).str.strip().isin(['0', '1'])]
df_trec_07['label'] = df_trec_07['label'].astype(int)
df_trec_07['label'] = (
    df_trec_07['label']
    .fillna(0)               
    .astype(str)             
    .str.strip()             
    .replace('', '0')        
    .astype(int)             
)
df_trec_05.head(3)

,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr PW: bnaweb22 -----O...,0,1
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,0
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,1


In [11]:
def count_email_labels(df, name="Dataset"):
    counts = df['label'].value_counts().to_dict()
    ham_count = counts.get(0, 0)
    spam_count = counts.get(1, 0)
    print(f"{name}\n")
    print(f"  Ham (label=0): {ham_count}")
    print(f"  Spam (label=1): {spam_count}")
    print(f"  Total emails: {ham_count + spam_count}\n")

    return {"ham": ham_count, "spam": spam_count}

In [12]:
count_email_labels(df_nazario, "Nazario")
count_email_labels(df_nigerian, "Nigerian Fraud")
count_email_labels(df_trec_05, "TREC_05")
count_email_labels(df_trec_06, "TREC_06")
count_email_labels(df_trec_07, "TREC_07")

Nazario

  Ham (label=0): 0
  Spam (label=1): 1565
  Total emails: 1565

Nigerian Fraud

  Ham (label=0): 0
  Spam (label=1): 3332
  Total emails: 3332

TREC_05

  Ham (label=0): 32278
  Spam (label=1): 22932
  Total emails: 55210

TREC_06

  Ham (label=0): 12393
  Spam (label=1): 3989
  Total emails: 16382

TREC_07

  Ham (label=0): 24353
  Spam (label=1): 29392
  Total emails: 53745



{'ham': 24353, 'spam': 29392}

In [13]:
df_nazario_safe = df_nazario[df_nazario['label'] == 0].copy()
df_nigerian_safe = df_nigerian[df_nigerian['label'] == 0].copy()
df_trec_05_safe = df_trec_05[df_trec_05['label'] == 0].copy()
df_trec_06_safe = df_trec_06[df_trec_06['label'] == 0].copy()
df_trec_07_safe = df_trec_07[df_trec_07['label'] == 0].copy()

In [14]:
df_nazario = df_nazario[df_nazario['label'] == 1].copy()
df_nigerian = df_nigerian[df_nigerian['label'] == 1].copy()
df_trec_05 = df_trec_05[df_trec_05['label'] == 1].copy()
df_trec_06 = df_trec_06[df_trec_06['label'] == 1].copy()
df_trec_07 = df_trec_07[df_trec_07['label'] == 1].copy()

In [15]:
df_nazario['category'] = 'phishing'
df_nigerian['category'] = 'phishing'
df_trec_05['category'] = 'spam'
df_trec_06['category'] = 'spam'
df_trec_07['category'] = 'spam'

In [16]:
safe_df = pd.concat([
    df_nazario_safe,
    df_nigerian_safe,
    df_trec_05_safe,
    df_trec_06_safe,
    df_trec_07_safe,
], ignore_index=True)

In [17]:
safe_df['category'] = 'legit'

In [18]:
safe_df.head(5)

,sender,receiver,date,subject,body,urls,label,category
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr PW: bnaweb22 -----O...,1,0.0,legit
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,0.0,legit
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,1,0.0,legit
3,"""Thorne, Judy"" <Judy.Thorne@ENRON.com>","""Grass, John"" <John.Grass@ENRON.com>, ""Nemec, ...","Fri, 29 Jun 2001 10:35:17 -0500",FW: ENA Upstream Company information,"John/Gerald, We are currently trading under GT...",0,0.0,legit
4,"""Williams, Jason R (Credit)"" <Jason.R.Williams...","""Nemec, Gerald"" <Gerald.Nemec@ENRON.com>, ""Dic...","Fri, 29 Jun 2001 10:40:02 -0500",New Master Physical,Gerald and Stacy -\n\nAttached is a worksheet ...,0,0.0,legit


In [19]:
safe_df.shape

(69024, 8)

In [20]:
df_combined = pd.concat([
    df_nazario,
    df_nigerian,
    df_trec_05,
    df_trec_06,
    df_trec_07,
    safe_df
], ignore_index=True)

In [21]:
df_combined.shape

(130234, 8)

In [22]:
df_combined.drop(columns=[c for c in ['label','date'] if c in df_combined.columns], inplace=True, errors='ignore')

In [23]:
df_combined.head(3)

,sender,receiver,subject,body,urls,category
0,Mail System Internal Data <MAILER-DAEMON@monke...,NaN,DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA,This text is part of the internal format of yo...,1,phishing
1,cPanel <service@cpanel.com>,jose@monkey.org,Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1,phishing
2,Microsoft Outlook <recepcao@unimedceara.com.br>,NaN,Helpdesk Mailbox Alert!!!,Your two incoming mails were placed on pending...,1,phishing


In [24]:
df_combined.to_csv('df_combined.csv', index=False)